In [4]:
import os
import pandas as pd
from PIL import Image

import sqlite3
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

In [5]:
class CosmosDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = int(self.df.iloc[idx]['label'])
        
        # Convert image to RGB format to handle grayscale or RGBA channels
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [6]:
# Data augmentation and normalization tailored for astronomical images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(180),  # Celestial objects have no fixed orientation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load custom dataset from CSV files
train_data = CosmosDataset(csv_file='cosmos_train.csv', transform=train_transform)
test_data = CosmosDataset(csv_file='cosmos_test.csv', transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

In [7]:
class_names = ['star', 'galaxy', 'quasar', 'nebula', 'planet']

print(f"Total training images: {len(train_data)}")
print(f"Total testing images: {len(test_data)}")

Total training images: 139
Total testing images: 35


In [8]:
# Select hardware acceleration (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device: {device}")

# Load pre-trained ResNet18 and adapt final fully-connected layer for 5 classes
net = resnet18(weights=ResNet18_Weights.DEFAULT)
num_ftrs = net.fc.in_features
net.fc = nn.Linear(num_ftrs, len(class_names))

net = net.to(device)

Training device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/rsalas/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


In [9]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0001)

In [10]:
epochs = 25  # Fine-tuning for 25 epochs to achieve >90% accuracy

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f} - Accuracy: {train_acc:.2f}%")

Epoch [1/25] - Loss: 1.3033 - Accuracy: 46.04%
Epoch [2/25] - Loss: 0.6738 - Accuracy: 80.58%
Epoch [3/25] - Loss: 0.4776 - Accuracy: 86.33%
Epoch [4/25] - Loss: 0.3863 - Accuracy: 86.33%
Epoch [5/25] - Loss: 0.2229 - Accuracy: 97.12%
Epoch [6/25] - Loss: 0.1718 - Accuracy: 97.12%
Epoch [7/25] - Loss: 0.1719 - Accuracy: 97.84%
Epoch [8/25] - Loss: 0.1175 - Accuracy: 98.56%
Epoch [9/25] - Loss: 0.0837 - Accuracy: 98.56%
Epoch [10/25] - Loss: 0.0941 - Accuracy: 98.56%
Epoch [11/25] - Loss: 0.1263 - Accuracy: 98.56%
Epoch [12/25] - Loss: 0.0338 - Accuracy: 100.00%
Epoch [13/25] - Loss: 0.0654 - Accuracy: 100.00%
Epoch [14/25] - Loss: 0.1203 - Accuracy: 97.84%
Epoch [15/25] - Loss: 0.0442 - Accuracy: 99.28%
Epoch [16/25] - Loss: 0.0333 - Accuracy: 100.00%
Epoch [17/25] - Loss: 0.0599 - Accuracy: 98.56%
Epoch [18/25] - Loss: 0.0338 - Accuracy: 100.00%
Epoch [19/25] - Loss: 0.0222 - Accuracy: 100.00%
Epoch [20/25] - Loss: 0.0216 - Accuracy: 100.00%
Epoch [21/25] - Loss: 0.0340 - Accuracy: 99

In [11]:
torch.save(net.state_dict(), 'trained_net.pth')
print("Model state successfully saved to 'trained_net.pth'")

Model state successfully saved to 'trained_net.pth'


In [12]:
# Evaluate model performance on unseen test data
net.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Final Model Test Accuracy: {accuracy:.2f}%")

UnidentifiedImageError: Caught UnidentifiedImageError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/rsalas/Documentos/NASA_TRAINING/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/rsalas/Documentos/NASA_TRAINING/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_10908/4252145174.py", line 14, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ^^^^^^^^^^^^^^^^^^^^
  File "/home/rsalas/Documentos/NASA_TRAINING/.venv/lib/python3.12/site-packages/PIL/Image.py", line 3715, in open
    raise UnidentifiedImageError(msg)
PIL.UnidentifiedImageError: cannot identify image file '/home/rsalas/Documentos/NASA_TRAINING/t

In [ ]:
import sqlite3
from datetime import datetime

mast_dir = '/home/rsalas/Documentos/NASA_TRAINING/images/mastDownload/HST'

def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = test_transform(image).unsqueeze(0).to(device)
    output = net(image)
    _, predicted = torch.max(output, 1)
    return class_names[predicted.item()]

# Connect to SQLite database (this creates the file if it doesn't exist)
db_path = 'mast_predictions.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create the table to store results
cursor.execute('''
    CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT,
        predicted_class TEXT,
        observation_id TEXT,
        timestamp TEXT
    )
''')
conn.commit()

print("CLASSIFYING HUBBLE / MAST IMAGES & SAVING TO DB")

if os.path.exists(mast_dir):
    for root, dirs, files in os.walk(mast_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                
                # Predict using the neural network
                prediction = predict_image(full_path)
                
                # Extract the folder name (MAST observation ID, e.g., 'iflh23zeq')
                observation_id = os.path.basename(root)
                current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                
                # Insert the record into the database
                cursor.execute('''
                    INSERT INTO predictions (filename, predicted_class, observation_id, timestamp)
                    VALUES (?, ?, ?, ?)
                ''', (file, prediction, observation_id, current_time))
                
                print(f"File: {file:<20} | Class: {prediction:<8} | DB: Saved")
    
    # Save changes and close connection
    conn.commit()
    conn.close()
    
    print(f"\nAll results successfully saved to database: '{db_path}'")
else:
    print(f"MAST directory not found at path: {mast_dir}")